In [2]:
! pip install langchain-openai dotenv -q

In [4]:
import asyncio
import base64
import json
import os
import wave
from pathlib import Path

import requests
from dotenv import load_dotenv
from IPython.display import Audio, display
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import AzureChatOpenAI

load_dotenv('.env', override=True)

True

In [6]:
llm = AzureChatOpenAI(
    azure_deployment=os.getenv('AZURE_DEPLOYMENT'),
    api_version=os.getenv('AZURE_OPENAI_API_VERSION'),
)

llm.invoke('hello')

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 7, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 3, 'engine_ttft_ms': 29, 'engine_ttlt_ms': 69, 'pre_inference_ms': 90, 'service_tbt_ms': 6, 'service_ttft_ms': 168, 'service_ttlt_ms': 255, 'total_duration_ms': 156, 'user_visible_ttft_ms': 78}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DbJfmfZD26HmB3TyyimoiZ8vi3dws', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filter

In [12]:
import numpy as np
import sounddevice as sd
from scipy.io import wavfile
from IPython.display import Audio, display

# 1. Show available microphone devices
print("Available input devices:\n")

input_devices = []
for index, device in enumerate(sd.query_devices()):
    if device["max_input_channels"] > 0:
        input_devices.append(index)
        print(
            f"{index}: {device['name']} "
            f"| inputs={device['max_input_channels']} "
            f"| default_sr={device['default_samplerate']}"
        )

# 2. Select microphone device ID
device_id = int(input("\nEnter microphone device ID: "))

# 3. Record for 10 seconds
duration_seconds = 10
sample_rate = 16000
output_path = "mic-recording.wav"

print(f"\nRecording from device {device_id} for {duration_seconds} seconds...")


recording = sd.rec(
    int(duration_seconds * sample_rate),
    samplerate=sample_rate,
    channels=1,
    dtype="float32",
    device=device_id,
)

sd.wait()

# 4. Save as WAV
audio_int16 = np.clip(recording.squeeze() * 32767, -32768, 32767).astype(np.int16)
wavfile.write(output_path, sample_rate, audio_int16)

print(f"Recording saved to: {output_path}")

# 5. Play audio in notebook
display(Audio(output_path))

Available input devices:

0: Microsoft Sound Mapper - Input | inputs=2 | default_sr=44100.0
1: Microphone Array (2- Intel® Sma | inputs=4 | default_sr=44100.0
2: Headset (EarFun Air Pro 3) | inputs=1 | default_sr=44100.0
3: CABLE Output (VB-Audio Virtual  | inputs=16 | default_sr=44100.0
4: Virtual Mic (Virtual Mic for Au | inputs=2 | default_sr=44100.0
5: Krisp Microphone (Krisp Audio) | inputs=1 | default_sr=44100.0
14: Primary Sound Capture Driver | inputs=2 | default_sr=44100.0
15: Microphone Array (2- Intel® Smart Sound Technology for Digital Microphones) | inputs=4 | default_sr=44100.0
16: Headset (EarFun Air Pro 3) | inputs=1 | default_sr=44100.0
17: CABLE Output (VB-Audio Virtual Cable) | inputs=16 | default_sr=44100.0
18: Virtual Mic (Virtual Mic for AudioRelay) | inputs=2 | default_sr=44100.0
19: Krisp Microphone (Krisp Audio) | inputs=1 | default_sr=44100.0
35: Headset (EarFun Air Pro 3) | inputs=1 | default_sr=16000.0
36: CABLE Output (VB-Audio Virtual Cable) | inputs=2 | d

In [ ]:
def get_audio_input(output_path="mic-recording.wav"):
    recording = sd.rec(
        int(duration_seconds * sample_rate),
        samplerate=sample_rate,
        channels=1,
        dtype="float32",
        device=device_id,
    )

    sd.wait()

    # 4. Save as WAV
    audio_int16 = np.clip(recording.squeeze() * 32767, -32768, 32767).astype(np.int16)
    wavfile.write(output_path, sample_rate, audio_int16)

    print(f"Recording saved to: {output_path}")

    # 5. Play audio in notebook
    display(Audio(output_path))

In [13]:
def speech_to_text(audio_path: str) -> str:
    url = 'https://api.elevenlabs.io/v1/speech-to-text'
    headers = {'xi-api-key': os.environ['ELEVENLABS_API_KEY']}
    data = {'model_id': os.getenv('ELEVENLABS_STT_MODEL', 'scribe_v2')}

    with open(audio_path, 'rb') as audio_file:
        files = {'file': (Path(audio_path).name, audio_file)}
        response = requests.post(url, headers=headers, data=data, files=files, timeout=120)

    response.raise_for_status()
    return response.json().get('text', '')

In [14]:
transcript = speech_to_text('mic-recording.wav')
print(transcript)

So tell me about AI agents, how do they work and what are different use cases they are used for and what frameworks are usually implemented


In [15]:
def text_to_speech(text: str, output_path: str) -> str:
    voice_id = os.environ['ELEVENLABS_VOICE_ID']
    model_id = os.getenv('ELEVENLABS_TTS_MODEL', 'eleven_multilingual_v2')
    url = f'https://api.elevenlabs.io/v1/text-to-speech/{voice_id}'

    response = requests.post(
        url,
        headers={
            'xi-api-key': os.environ['ELEVENLABS_API_KEY'],
            'Content-Type': 'application/json',
            'Accept': 'audio/mpeg',
        },
        json={'text': text, 'model_id': model_id},
        timeout=120,
    )
    response.raise_for_status()

    with open(output_path, 'wb') as audio_file:
        audio_file.write(response.content)
    return output_path

In [16]:
text_to_speech("AI Agents are llms with tool calling capabilities", "response.mp3")

'response.mp3'

In [ ]:
while True:
    input_audio = 'mic-recording.wav'
    get_audio_input(output_path=input_audio)

    transcript = speech_to_text(input_audio)
    print('Transcript:', transcript)

    prompt = "Answer the following in 50 words or less: " + transcript

    llm_response = llm.invoke(prompt)
    print('LLM Response:', llm_response)

    text_to_speech(llm_response.content, "response.mp3")
    display(Audio("response.mp3"))

Transcript: So tell me about AI agents, how do they work and what are different use cases they are used for and what frameworks are usually implemented
LLM Response: content='AI agents are software that perceive context, decide actions, and use tools to complete tasks with minimal human input. They often combine an LLM, memory, planning, and tool-calling. Use cases: customer support, research, coding, workflow automation, data analysis, and assistants. Common frameworks: LangChain, LlamaIndex, AutoGen, CrewAI, and Semantic Kernel.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 43, 'total_tokens': 124, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 5, 'engine_ttft_ms': 33, 'engine_ttlt_ms': 415, 'pre_inference_ms': 129, 'servi